In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
data=pd.read_csv("data.csv")

In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 61692 entries, 0 to 61691
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Text        61121 non-null  object
 1   Labels      61692 non-null  int64 
 2   clean_text  60814 non-null  object
 3   hashtags    61692 non-null  object
 4   mentions    61692 non-null  object
dtypes: int64(1), object(4)
memory usage: 2.4+ MB


In [4]:
data.head()

,Text,Labels,clean_text,hashtags,mentions
0,im getting on borderlands and i will murder yo...,2,im getting on borderlands and i will murder yo...,[],[]
1,I am coming to the borders and I will kill you...,2,i am coming to the borders and i will kill you...,[],[]
2,im getting on borderlands and i will kill you ...,2,im getting on borderlands and i will kill you all,[],[]
3,im coming on borderlands and i will murder you...,2,im coming on borderlands and i will murder you...,[],[]
4,im getting on borderlands 2 and i will murder ...,2,im getting on borderlands and i will murder yo...,[],[]


In [5]:
data.isna().sum()

Text          571
Labels          0
clean_text    878
hashtags        0
mentions        0
dtype: int64

In [6]:
data.dropna(inplace=True)

In [7]:
data.duplicated().sum()

np.int64(3455)

In [8]:
data.drop_duplicates(inplace=True)

In [9]:
X=data["clean_text"]
y=data["Labels"]

In [10]:
from sklearn.model_selection import train_test_split
Xtrain,Xtest,ytrain,ytest=train_test_split(X,y,test_size=0.2,stratify=y,random_state=42)

In [11]:
"""from imblearn.over_sampling import SMOTE

sm = SMOTE(random_state=42)
Xtrain_re, ytrain_re = sm.fit_resample(X, y)
"""

'from imblearn.over_sampling import SMOTE\n\nsm = SMOTE(random_state=42)\nXtrain_re, ytrain_re = sm.fit_resample(X, y)\n'

In [12]:
Xtrain.shape

(45887,)

In [13]:
data["Labels"].value_counts()

Labels
0    21192
2    19107
1    17060
Name: count, dtype: int64

In [14]:
from tensorflow.keras.preprocessing.text import Tokenizer
tokenizer = Tokenizer()
tokenizer.fit_on_texts(Xtrain)

# 3) Transform train & test
X_train_seq = tokenizer.texts_to_sequences(Xtrain)
X_test_seq = tokenizer.texts_to_sequences(Xtest)

In [15]:
max_len = max(len(x) for x in X_train_seq)

In [16]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len)
X_test_pad  = pad_sequences(X_test_seq, maxlen=max_len)


In [17]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    'balanced',
    classes=np.unique(ytrain),
    y=ytrain
)

class_weights = dict(enumerate(class_weights))


In [18]:
X_train_pad

array([[    0,     0,     0, ...,    10,    53,   444],
       [    0,     0,     0, ...,    60,  6847,   163],
       [    0,     0,     0, ...,   139,    33,     7],
       ...,
       [    0,     0,     0, ...,   562,  9885,  4707],
       [    0,     0,     0, ...,   105,  1821, 11316],
       [    0,     0,     0, ...,   165,  2815, 16153]],
      shape=(45887, 166), dtype=int32)

In [19]:
vocab_size = len(tokenizer.word_index) + 1

In [20]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, SimpleRNN, Dense, Dropout
from tensorflow.keras.regularizers import l1_l2

reg = l1_l2(l1=1e-5, l2=1e-4)   # You can tune these values

# LSTM model with L1 & L2
model_LSTM = Sequential([
    Embedding(input_dim=vocab_size, output_dim=128, input_length=max_len),

    LSTM(
        128,
        activation="tanh",
        return_sequences=True,
        dropout=0.1,
        recurrent_dropout=0.1,
        kernel_regularizer=reg,
        recurrent_regularizer=reg
    ),

    LSTM(
        256,
        activation="tanh",
        return_sequences=True,
        dropout=0.1,
        recurrent_dropout=0.1,
        kernel_regularizer=reg,
        recurrent_regularizer=reg
    ),

    LSTM(
        256,
        activation="tanh",
        return_sequences=True,
        dropout=0.1,
        recurrent_dropout=0.1,
        kernel_regularizer=reg,
        recurrent_regularizer=reg
    ),

    LSTM(
        64,
        activation="tanh",
        dropout=0.1,
        recurrent_dropout=0.1,
        kernel_regularizer=reg,
        recurrent_regularizer=reg
    ),

    Dense(32, activation="tanh", kernel_regularizer=reg),
    Dropout(0.5),

    Dense(3, activation="softmax", kernel_regularizer=reg)
])


# RNN model with L1 & L2
model_RNN = Sequential([
    Embedding(input_dim=vocab_size, output_dim=128, input_length=max_len),

    SimpleRNN(
        128,
        activation="tanh",
        return_sequences=True,
        dropout=0.1,
        recurrent_dropout=0.1,
        kernel_regularizer=reg,
        recurrent_regularizer=reg
    ),

    SimpleRNN(
        256,
        activation="tanh",
        return_sequences=True,
        dropout=0.1,
        recurrent_dropout=0.1,
        kernel_regularizer=reg,
        recurrent_regularizer=reg
    ),

    SimpleRNN(
        256,
        activation="tanh",
        return_sequences=True,
        dropout=0.1,
        recurrent_dropout=0.1,
        kernel_regularizer=reg,
        recurrent_regularizer=reg
    ),

    SimpleRNN(
        64,
        activation="tanh",
        dropout=0.1,
        recurrent_dropout=0.1,
        kernel_regularizer=reg,
        recurrent_regularizer=reg
    ),

    Dense(32, activation="tanh", kernel_regularizer=reg),
    Dropout(0.5),

    Dense(3, activation="softmax", kernel_regularizer=reg)
])


c:\Users\mohma\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [21]:
model_LSTM.compile(optimizer="adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"])
model_LSTM.summary()

model_RNN.compile(optimizer="adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"])
model_RNN.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_2 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_3 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history_LSTM = model_LSTM.fit(
    X_train_pad,
    ytrain,
    epochs=5,
    batch_size=128,
    validation_split=0.2,  # <-- split 10% of training data for validation
    class_weight=class_weights
)
history_RNN = model_RNN.fit(
    X_train_pad,
    ytrain,
    epochs=7,
    batch_size=128,
    validation_split=0.2,  # <-- use your validation dataset
    class_weight=class_weights
)


Epoch 1/5
287/287 ━━━━━━━━━━━━━━━━━━━━ 787s 3s/step - accuracy: 0.6082 - loss: 1.0052 - val_accuracy: 0.7465 - val_loss: 0.7127
Epoch 2/5
 22/287 ━━━━━━━━━━━━━━━━━━━━ 11:10:45 152s/step - accuracy: 0.8073 - loss: 0.6283

In [ ]:
model_RNN.evaluate(X_test_pad, ytest)

359/359 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.8114 - loss: 0.7297


[0.7297012805938721, 0.8113667964935303]

In [ ]:
model_LSTM.evaluate(X_test_pad, ytest)

359/359 ━━━━━━━━━━━━━━━━━━━━ 10s 29ms/step - accuracy: 0.8913 - loss: 0.3709


[0.3709057569503784, 0.8913005590438843]

In [ ]:
import re
def clean_text(text):
    text = str(text).lower() # Hello World! → hello world!
    text = re.sub(r"http\S+", "", text) # http://example.com → ""
    text = re.sub(r"@\w+", "", text) # @user123 → ""
    text = re.sub(r"#", "", text) # "#iphone" → "iphone"
    text = re.sub(r"[^a-zA-Z ]", " ", text) # hello!!! → hello
    text = re.sub(r"\s+", " ", text).strip() #"hello     world  " → "hello world"
    return text

def predict(text):
    seq = tokenizer.texts_to_sequences([clean_text(text)])
    seq = pad_sequences(seq, maxlen=max_len)
    
    pred = model_LSTM.predict(seq)
    labels = ["Negative", "Neutral", "Positive"]
    return labels[pred.argmax()]

predict("I love this phone!")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 394ms/step


'Negative'

In [ ]:
model_LSTM.save('model_LSTM.h5')

In [ ]:
model_RNN.save('model_RNN.h5')

In [ ]:
from tensorflow.keras.models import load_model
model=load_model("model_RNN.h5")

In [ ]:
import re
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
max_len = max(len(x) for x in X_train_seq)
tokenizer = Tokenizer()
def clean_text(text):
    text = str(text).lower() # Hello World! → hello world!
    text = re.sub(r"http\S+", "", text) # http://example.com → ""
    text = re.sub(r"@\w+", "", text) # @user123 → ""
    text = re.sub(r"#", "", text) # "#iphone" → "iphone"
    text = re.sub(r"[^a-zA-Z ]", " ", text) # hello!!! → hello
    text = re.sub(r"\s+", " ", text).strip() #"hello     world  " → "hello world"
    return text

def predict(text):
    seq = tokenizer.texts_to_sequences([clean_text(text)])
    seq = pad_sequences(seq, maxlen=max_len)
    
    pred = model.predict(seq)
    labels = ["Negative", "Neutral", "Positive"]
    return labels[pred.argmax()] , pred
predict("""I hate thinking that this easy mayhem modifier event on total mayhem won ’ t last forever. this is the absolutely most ridiculous fun experience i ’ ve had in the game almost since... they added them horrible modifiers. @Borderlands please do give me the option option to a play mayhem 10 but turn the modifiers off PLS,1,i hate thinking that this easy mayhem modifier event on total mayhem won t last forever this is the absolutely most ridiculous fun experience i ve had in the game almost since they added them horrible modifiers please do give me the option option to a play mayhem but turn the modifiers off pls,[],['Borderlands']
I hate that this easy horrible modifier event on game 10 last forever. That is the best fun i’ve taken in the past until they added them horrible modifiers. @Borderlands please send me the option to play mayhem 10 but turn the modifiers for PLS,1,i hate that this easy horrible modifier event on game last forever that is the best fun i ve taken in the past until they added them horrible modifiers please send me the option to play mayhem but turn the modifiers for pls,[],['Borderlands']
""")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step


('Negative', array([[0.7634853 , 0.05324227, 0.18327242]], dtype=float32))